# Exp 18: Grid dropout

Card: [exp18_grid_dropout.md](../../docs/experiments/exp18_grid_dropout.md)


In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from core.data.augment import Validation
from core.training.config import ROOT, SEED, TRAIN_DIR
from core.training.config import VIP
from core.training.cli import data_module
from core.training.cv import CVResult
from core.training.runs import read_config
from core.analysis.gradcam import GradCamFigures
from core.analysis.plots import to_image, pr_curves, save_figures, use_style

N_IMAGES = 8
MAX_DRAWS = None  # all draws
N_POSITIVES = None  # all labeled positives
CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp18_grid_dropout" / "resnet18.yaml"
EXP16 = CONFIG_DIR / "exp16_downsampling" / "resnet18.yaml"
FOLD = 0
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp18_grid_dropout"

# dataviz reference palette, light mode
SERIES = {"exp16": "#eb6834", "grid dropout": "#2a78d6"}
use_style()

## Oversampled views (exp13)


In [ ]:
torch.manual_seed(SEED)
dm = data_module(CONFIG, FOLD)
dm.setup()
train = dm.train_ds.df
draws = Counter(dm.train_dataloader().sampler)  # dataset row: times drawn in epoch


resize = Validation(dm.hparams.img_size).compose()
rng = np.random.default_rng(SEED)
figures = {}
for cls in VIP:
    rows = rng.choice(train.index[train[cls] == 1], N_IMAGES, replace=False)
    n_cols = 1 + (MAX_DRAWS or max(draws[i] for i in rows))
    fig, axes = plt.subplots(
        N_IMAGES, n_cols, figsize=(1.8 * n_cols, 2.0 * N_IMAGES), squeeze=False
    )
    for ax_row, i in zip(axes, rows):
        image = np.array(
            Image.open(TRAIN_DIR / f"{train.loc[i, 'image_id']}.jpg").convert("RGB")
        )
        ax_row[0].imshow(to_image(resize(image=image)["image"]))
        ax_row[0].set_title(f"original · {draws[i]} draws", fontsize=8)
        for ax in ax_row[1 : 1 + min(draws[i], n_cols - 1)]:
            ax.imshow(
                to_image(dm.train_ds[i][0])
            )  # fresh augmentation per draw, as in training
        for ax in ax_row:
            ax.set_axis_off()
    fig.suptitle(
        f"{cls} · fold {FOLD} train · one epoch, oversample {dm.oversample}, "
        f"downsample {dm.downsample}, augment {read_config(CONFIG)['data'].get('augment')}"
    )
    fig.tight_layout()
    figures[f"{PREFIX}_views_{cls.lower()}.png"] = fig
plt.to_image()

## Training


In [ ]:
results = {"exp16": CVResult.load(EXP16)}
results["grid dropout"] = CVResult.load(CONFIG)

## Precision-recall curves


In [ ]:
fig = pr_curves(results, colors=SERIES)
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.to_image()

## Grad-CAM VIP: unseen vs trained images


In [ ]:
cams = GradCamFigures(CONFIG)
for cls in VIP:
    for train in (False, True):
        fig = cams.positives(cls, train=train, n=N_POSITIVES)
        split = "train" if train else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.to_image()

In [ ]:
save_figures(figures, FIGURES)